In [1]:
import sympy as sp
from itertools import product

def run_rom_peters_search():
    # 1. Define symbolic variables
    e, o = sp.symbols('e o', real=True)

    # 2. R.O.M. Structural Intensities
    # (Extracting only the phase-dependent polynomial numerators for the search)
    B2 = 1 + e**2 + 2*e*sp.cos(o)       # beta_o^2
    BT2 = (1 + e*sp.cos(o))**2          # beta_T^2 (Transverse/Delay intensity)
    Q2 = 3 + e**2 + 4*e*sp.cos(o)       # Q_o^2
    K2 = 2*(1 + e*sp.cos(o))            # kappa_o^2

    # 3. Unitary State Projections (Closure Defect Candidates)
    Tau_Y2 = Q2 - K2 * B2               # tau_yo^2
    Tau2 = 1 - Tau_Y2                   # tau_o^2

    # 4. Integration Weight (Inverse angular frequency denominator structure)
    W = 1 / (1 + e*sp.cos(o))**2

    # 5. Target Peters Polynomial Coefficients
    peters_target = [1, sp.Rational(73, 24), sp.Rational(37, 96)]

    # Dictionary of available operators to combine
    operators = {
        'B2': B2,
        'BT2': BT2,
        'Q2': Q2,
        'K2': K2,
        'Tau_Y2': Tau_Y2,
        'Tau2': Tau2
    }

    keys = list(operators.keys())
    max_power_per_term = 2 # Adjust depth of search here

    print("Executing R.O.M. Algebraic Search for Peters Polynomial...")
    print(f"Target vector: {peters_target}\n")

    # 6. Combinatorial Search Space
    # Generates all combinations of powers: e.g., (B2^1 * BT2^1 * Q2^0 ...)
    for powers in product(range(max_power_per_term + 1), repeat=len(keys)):
        if sum(powers) == 0:
            continue

        # Build the candidate expression
        candidate_expr = 1
        term_names = []
        for key, power in zip(keys, powers):
            if power > 0:
                candidate_expr *= operators[key]**power
                term_names.append(f"{key}^{power}" if power > 1 else key)

        candidate_name = " * ".join(term_names)

        # 7. Apply time-weighting and integrate over topological cycle [0, 2*pi]
        integrand = sp.expand(candidate_expr * W)

        # Skip if the integrand contains an unresolved denominator (would yield an elliptic integral)
        if not integrand.is_polynomial(sp.cos(o)):
            continue

        # Integrate (sympy handles trigonometric integration perfectly)
        integral_result = sp.integrate(integrand, (o, 0, 2*sp.pi))

        # 8. Coefficient Extraction and Normalization
        # Extract polynomial coefficients for e
        poly = sp.Poly(integral_result, e)
        coeffs_dict = poly.as_dict()

        # Ensure it contains only even powers of e (0, 2, 4) up to e^4
        # and doesn't contain higher powers for a strict match
        valid_structure = True
        extracted_vector = [0, 0, 0] # for e^0, e^2, e^4

        for power_tuple, coeff in coeffs_dict.items():
            power = power_tuple[0]
            if power == 0:
                extracted_vector[0] = coeff
            elif power == 2:
                extracted_vector[1] = coeff
            elif power == 4:
                extracted_vector[2] = coeff
            else:
                valid_structure = False
                break

        if not valid_structure or extracted_vector[0] == 0:
            continue

        # Normalize the vector to compare with Peters (make e^0 coefficient = 1)
        base_val = extracted_vector[0]
        normalized_vector = [c / base_val for c in extracted_vector]

        # 9. Verification
        if normalized_vector == peters_target:
            print(f"[EXACT MATCH FOUND]")
            print(f"Operator: {candidate_name}")
            print(f"Raw Integral: {integral_result}")
            print("-" * 50)
            return

    print("Search completed. No exact match found in current depth/configuration.")
    print("Consider expanding max_power_per_term or adding cross-terms (e.g. beta_A * beta_B).")

if __name__ == "__main__":
    run_rom_peters_search()

Executing R.O.M. Algebraic Search for Peters Polynomial...
Target vector: [1, 73/24, 37/96]

Search completed. No exact match found in current depth/configuration.
Consider expanding max_power_per_term or adding cross-terms (e.g. beta_A * beta_B).


In [5]:
import sympy as sp
from itertools import product
import time
import sys

def run_safe_rom_search():
    e, o = sp.symbols('e o', real=True)

    # 1. Structural Intensities R.O.M. (Only Numerators of phase)
    B2 = 1 + e**2 + 2*e*sp.cos(o)       # beta_o^2
    BT2 = (1 + e*sp.cos(o))**2          # beta_T^2 (Must be present to cancel W!)
    Q2 = 3 + e**2 + 4*e*sp.cos(o)       # Q_o^2
    K2 = 2*(1 + e*sp.cos(o))            # kappa_o^2
    Tau_Y2 = Q2 - K2 * B2               # tau_yo^2

    # 2. Integration Weight Denominator
    W = 1 / (1 + e*sp.cos(o))**2

    # 3. Target: Peters Polynomial [1, 73/24, 37/96]
    peters_target = [1, sp.Rational(73, 24), sp.Rational(37, 96)]

    operators = {'B2': B2, 'BT2': BT2, 'Q2': Q2, 'K2': K2, 'Tau_Y2': Tau_Y2}
    keys = list(operators.keys())
    max_power = 2 # Degree of combinations

    total_combinations = (max_power + 1)**len(keys) - 1

    print("Executing SAFE R.O.M. Algebraic Search...")
    print(f"Target vector: {peters_target}")
    print(f"Total combinations to check: {total_combinations}\n")

    start_time = time.time()
    checked = 0
    valid_integrals_calculated = 0

    for powers in product(range(max_power + 1), repeat=len(keys)):
        if sum(powers) == 0:
            continue

        checked += 1
        if checked % 50 == 0:
            elapsed = time.time() - start_time
            sys.stdout.write(f"\r[Progress: {checked}/{total_combinations}] | Pure Polynomials Found: {valid_integrals_calculated} | Elapsed: {elapsed:.1f}s")
            sys.stdout.flush()

        candidate_expr = 1
        term_names = []
        for key, power in zip(keys, powers):
            if power > 0:
                candidate_expr *= operators[key]**power
                term_names.append(f"{key}^{power}" if power > 1 else key)

        candidate_name = " * ".join(term_names)

        # APPLY WEIGHT
        raw_integrand = candidate_expr * W

        # ALGEBRAIC SAFETY FILTER (The Anti-Hang Mechanism)
        simplified = sp.cancel(raw_integrand)
        denom = sp.denom(simplified)

        # If 'o' is still in the denominator, it's an elliptic trap. Skip!
        if o in denom.free_symbols:
            continue

        valid_integrals_calculated += 1

        # Now it is perfectly safe to integrate
        integral_result = sp.integrate(simplified, (o, 0, 2*sp.pi))

        # Extract coefficients
        poly = sp.Poly(integral_result, e).as_dict()
        extracted = [poly.get((0,), 0), poly.get((2,), 0), poly.get((4,), 0)]

        # Check if it has unsupported higher powers (e.g., e^6)
        if any(k[0] not in (0, 2, 4) for k in poly.keys()):
            continue

        if extracted[0] == 0:
            continue

        normalized = [c / extracted[0] for c in extracted]

        if normalized == peters_target:
            print("\n" + "="*60)
            print("💎 EXACT MATCH FOUND! 💎")
            print(f"Operator: {candidate_name}")
            print(f"Raw Integral: {integral_result}")
            print("="*60 + "\n")
            return

    total_time = time.time() - start_time
    print(f"\n\nSearch completed in {total_time:.1f}s.")
    print(f"Calculated {valid_integrals_calculated} safe polynomials out of {total_combinations} combinations.")
    print("No exact match found. The correct operator requires cross-terms or a specific ratio.")

if __name__ == "__main__":
    run_safe_rom_search()

Executing SAFE R.O.M. Algebraic Search...
Target vector: [1, 73/24, 37/96]
Total combinations to check: 242

[Progress: 200/242] | Pure Polynomials Found: 146 | Elapsed: 24.7s

Search completed in 37.6s.
Calculated 189 safe polynomials out of 242 combinations.
No exact match found. The correct operator requires cross-terms or a specific ratio.


In [1]:
import math

def run_numerical_calibration():
    # 1. Observables for Hulse-Taylor (PSR B1913+16)
    # Strictly using what reaches the detector (Epistemic Hygiene)
    c = 299792.458                 # km/s
    e = 0.6171334                  # Eccentricity
    K_A = 198.83                   # Pulsar semi-amplitude (km/s)
    K_B = 206.11                   # Companion semi-amplitude (km/s)
    sin_i = 0.7319                 # Inclination geometry (sin(47.05 deg))

    # Observed Decay Rate to match
    target_P_dot = 2.4225e-12

    print("--- WILL RG: Numerical Calibration for PSR B1913+16 ---")
    print(f"Target P_dot: {target_P_dot:.4e} s/s\n")

    # 2. Relational Invariants (Conversion from observables)
    # Total Kinematic Intensity (True beta, adjusting for inclination)
    beta_Sigma = (K_A + K_B) * math.sqrt(1 - e**2) / (c * sin_i)

    # Relational Symmetry Factor (Cross-term proportion without mass)
    eta_R = (K_A * K_B) / ((K_A + K_B)**2)

    # Eccentricity Form Factor (The separated Peters Polynomial)
    f_e = 1 + (73/24)*e**2 + (37/96)*e**4

    print(f"[Calculated Invariants]")
    print(f"Kinematic Intensity (beta_Sigma): {beta_Sigma:.6e}")
    print(f"Symmetry Factor (eta_R): {eta_R:.6f}")
    print(f"Geometric Form Factor f(e): {f_e:.5f}\n")

    # 3. Unitary Differential Bridge Multiplier
    # dP/dt = Bridge_Multiplier * Delta_tau
    bridge_multiplier = 3 / (2 * beta_Sigma**2 * (3 - 4 * beta_Sigma**2))

    # 4. Combinatorial Search Space
    topo_factors = {
        "1": 1,
        "pi": math.pi,
        "2*pi": 2 * math.pi,
        "4*pi": 4 * math.pi,
        "8*pi": 8 * math.pi,
        "16*pi": 16 * math.pi,
        "32*pi": 32 * math.pi,
        "64*pi": 64 * math.pi
    }

    eta_powers = [1, 2] # eta_R or eta_R^2
    beta_powers = [4, 5, 6, 7, 8, 9, 10] # Degrees of causal delay

    results = []

    # 5. Execute Calibration Loop
    for topo_name, topo_val in topo_factors.items():
        for p_eta in eta_powers:
            for p_beta in beta_powers:

                # Assemble the Closure Defect (Delta tau_cycle)
                delta_tau = topo_val * (eta_R**p_eta) * (beta_Sigma**p_beta) * f_e

                # Pass through the Unitary Bridge
                P_dot_calc = bridge_multiplier * delta_tau

                # Calculate Error
                error_pct = abs(P_dot_calc - target_P_dot) / target_P_dot * 100

                # Store structural formula name
                operator_name = f"{topo_name} * (\u03B7_R)^{p_eta} * (\u03B2_\u03A3)^{p_beta} * f(e)"

                results.append({
                    "formula": operator_name,
                    "p_dot": P_dot_calc,
                    "error": error_pct
                })

    # 6. Sort and Display the Top 5 Most Probable Architectures
    results.sort(key=lambda x: x['error'])

    print("--- Top 5 Most Probable Geometric Assemblies ---")
    for i, res in enumerate(results[:5]):
        print(f"Rank {i+1}:")
        print(f"  Operator \u0394\u03C4  = {res['formula']}")
        print(f"  Calculated P_dot = {res['p_dot']:.4e} s/s")
        print(f"  Accuracy Error   = {res['error']:.2f}%\n")

if __name__ == "__main__":
    run_numerical_calibration()

--- WILL RG: Numerical Calibration for PSR B1913+16 ---
Target P_dot: 2.4225e-12 s/s

[Calculated Invariants]
Kinematic Intensity (beta_Sigma): 1.452161e-03
Symmetry Factor (eta_R): 0.249919
Geometric Form Factor f(e): 2.21433

--- Top 5 Most Probable Geometric Assemblies ---
Rank 1:
  Operator Δτ  = 2*pi * (η_R)^2 * (β_Σ)^6 * f(e)
  Calculated P_dot = 1.9322e-12 s/s
  Accuracy Error   = 20.24%

Rank 2:
  Operator Δτ  = 1 * (η_R)^1 * (β_Σ)^6 * f(e)
  Calculated P_dot = 1.2305e-12 s/s
  Accuracy Error   = 49.21%

Rank 3:
  Operator Δτ  = 4*pi * (η_R)^2 * (β_Σ)^6 * f(e)
  Calculated P_dot = 3.8644e-12 s/s
  Accuracy Error   = 59.52%

Rank 4:
  Operator Δτ  = pi * (η_R)^1 * (β_Σ)^6 * f(e)
  Calculated P_dot = 3.8657e-12 s/s
  Accuracy Error   = 59.57%

Rank 5:
  Operator Δτ  = pi * (η_R)^2 * (β_Σ)^6 * f(e)
  Calculated P_dot = 9.6610e-13 s/s
  Accuracy Error   = 60.12%

